# Breakout Strategy

Trend-following strategy using moving average crossovers.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Configuration
SYMBOL = "AAPL"
START_DATE = "2023-01-01"
END_DATE = "2024-01-01"
SHORT_MA = 20
LONG_MA = 50
BREAKOUT_THRESHOLD = 0.02  # 2% breakout
INITIAL_CAPITAL = 100000

In [ ]:
# Fetch data (placeholder - replace with actual data source)
def fetch_data(symbol: str, start: str, end: str):
    """Fetch historical price data"""
    # TODO: Integrate with CBSC data service
    dates = pd.date_range(start=start, end=end, freq='D')
    np.random.seed(42)
    prices = np.random.randn(len(dates)).cumsum() + 100

    return pd.DataFrame({
        'date': dates,
        'open': prices,
        'high': prices * 1.02,
        'low': prices * 0.98,
        'close': prices,
        'volume': np.random.randint(1000000, 10000000, len(dates))
    }).set_index('date')

data = fetch_data(SYMBOL, START_DATE, END_DATE)
print(f"Loaded {len(data)} days of data for {SYMBOL}")

In [ ]:
# Calculate indicators
data['ma_short'] = data['close'].rolling(window=SHORT_MA).mean()
data['ma_long'] = data['close'].rolling(window=LONG_MA).mean()

# Generate signals
data['signal'] = 0
data.loc[
    (data['ma_short'] > data['ma_long']) &
    (data['close'] > data['ma_short'] * (1 + BREAKOUT_THRESHOLD)),
    'signal'
] = 1

data.loc[
    (data['ma_short'] < data['ma_long']),
    'signal'
] = -1

print(f"Generated {data['signal'].abs().sum()} trading signals")

In [ ]:
# Backtest
position = 0
cash = INITIAL_CAPITAL
portfolio_values = []

for i, row in data.iterrows():
    if row['signal'] == 1 and position == 0:
        # Buy
        position = cash / row['close']
        cash = 0
    elif row['signal'] == -1 and position > 0:
        # Sell
        cash = position * row['close']
        position = 0

    portfolio_values.append(cash + position * row['close'])

data['portfolio_value'] = portfolio_values
data['returns'] = data['portfolio_value'].pct_change()

In [ ]:
# Calculate performance metrics
total_return = (data['portfolio_value'].iloc[-1] / INITIAL_CAPITAL - 1) * 100
sharpe_ratio = data['returns'].mean() / data['returns'].std() * np.sqrt(252) if data['returns'].std() > 0 else 0
max_drawdown = (data['portfolio_value'].cummax() - data['portfolio_value']).max() / INITIAL_CAPITAL * 100

print(f"Total Return: {total_return:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2f}%")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Price and moving averages
ax1.plot(data.index, data['close'], label='Close', alpha=0.7)
ax1.plot(data.index, data['ma_short'], label=f'{SHORT_MA}-day MA', alpha=0.7)
ax1.plot(data.index, data['ma_long'], label=f'{LONG_MA}-day MA', alpha=0.7)
ax1.set_title(f'{SYMBOL} Breakout Strategy')
ax1.set_ylabel('Price')
ax1.legend()

# Portfolio value
ax2.plot(data.index, data['portfolio_value'], label='Portfolio', color='green')
ax2.set_title('Portfolio Performance')
ax2.set_ylabel('Portfolio Value ($)')
ax2.legend()

plt.tight_layout()
plt.show()